# Manual Reference Card Splitter (Transparent PNG + CSV Sync)

This notebook processes manual masks from data/iapr-26-uno-vision-challenge/reference_manual_mask/.

For each mask image, it exports:
- JPG card crops (for existing loaders)
- Binary JPG masks
- RGBA PNG crops where background is fully transparent

It also synchronizes project/training_data/object_labels/reference_cards/reference_manual.csv on each run:
- existing labels are preserved
- new crops are added
- removed crops are dropped

In [1]:
from pathlib import Path
import sys

candidate_src_dirs = [
    Path.cwd() / "src",
    Path.cwd().parent.parent / "src",
]
for src_dir in candidate_src_dirs:
    if src_dir.is_dir():
        parent = src_dir.parent.resolve()
        if str(parent) not in sys.path:
            sys.path.insert(0, str(parent))
        break
else:
    raise FileNotFoundError("Could not locate src directory for imports.")

from src.create_reference_cards import (
    ManualReferenceConfig,
    initialize_manual_reference_pipeline,
    run_manual_reference_split,
    summarize_manual_reference_split,
)

In [2]:
# Set image_names=None to process all manual masks in reference_manual_mask/
CFG = ManualReferenceConfig(
    image_names=None,
    preview_image_name="L1000765",
    mask_threshold=127,
    min_component_area_abs=1500,
    output_tag_suffix="",
    enforce_portrait_orientation=True,
    write_jpg_crops=True,
    write_binary_masks=True,
    write_transparent_png_crops=True,
    manual_reference_csv_name="reference_manual.csv",
    seed_from_reference_csv=True,
    seed_reference_csv_name="reference_do.csv",
)

CFG

ManualReferenceConfig(image_names=None, preview_image_name='L1000765', mask_threshold=127, min_component_area_abs=1500, row_group_tolerance_fraction=0.6, row_group_min_tolerance_px=10, enforce_portrait_orientation=True, output_tag_suffix='', clean_previous_outputs=True, write_jpg_crops=True, write_binary_masks=True, write_transparent_png_crops=True, write_indexed_previews=True, preview_subdir_name='previews', manual_reference_csv_name='reference_manual.csv', seed_from_reference_csv=True, seed_reference_csv_name='reference_do.csv', write_components_csv=True, manual_masks_subpath=('data', 'iapr-26-uno-vision-challenge', 'reference_manual_mask'), reference_images_subpath=('data', 'iapr-26-uno-vision-challenge', 'reference_images'), reference_cards_subpath=('project', 'training_data', 'training_images', 'reference_cards'), reference_labels_subpath=('project', 'training_data', 'object_labels', 'reference_cards'))

## Initialize

In [3]:
state = initialize_manual_reference_pipeline(CFG)
state["selected_image_names"]

Project root: /Users/dorianbesson/EDOC/IA/iapr2026do
Manual masks: /Users/dorianbesson/EDOC/IA/iapr2026do/data/iapr-26-uno-vision-challenge/reference_manual_mask
Reference images: /Users/dorianbesson/EDOC/IA/iapr2026do/data/iapr-26-uno-vision-challenge/reference_images
Reference output root: /Users/dorianbesson/EDOC/IA/iapr2026do/project/training_data/training_images/reference_cards
Label output dir: /Users/dorianbesson/EDOC/IA/iapr2026do/project/training_data/object_labels/reference_cards
Selected images: ['L1000765', 'L1000766', 'L1000767', 'L1000768']


['L1000765', 'L1000766', 'L1000767', 'L1000768']

## Split Masks + Sync Labels CSV

In [4]:
state = run_manual_reference_split(state)
summary = summarize_manual_reference_split(state)
summary

L1000765: 12 components | jpg=12 masks=12 rgba=12
L1000766: 12 components | jpg=12 masks=12 rgba=12
L1000767: 14 components | jpg=14 masks=14 rgba=14
L1000768: 16 components | jpg=16 masks=16 rgba=16
reference_manual.csv sync: rows=54 | prefilled=54 | empty=0
Reference manual CSV: /Users/dorianbesson/EDOC/IA/iapr2026do/project/training_data/object_labels/reference_cards/reference_manual.csv
Reference components CSV: /Users/dorianbesson/EDOC/IA/iapr2026do/project/training_data/object_labels/reference_cards/reference_manual_components.csv


{'images': {'L1000765': {'components': 12,
   'jpg_crops': 12,
   'jpg_masks': 12,
   'rgba_png_crops': 12},
  'L1000766': {'components': 12,
   'jpg_crops': 12,
   'jpg_masks': 12,
   'rgba_png_crops': 12},
  'L1000767': {'components': 14,
   'jpg_crops': 14,
   'jpg_masks': 14,
   'rgba_png_crops': 14},
  'L1000768': {'components': 16,
   'jpg_crops': 16,
   'jpg_masks': 16,
   'rgba_png_crops': 16}},
 'total_images': 4,
 'total_components': 54,
 'rows_written': 54,
 'newly_unlabeled': 0,
 'prefilled_from_seed': 54,
 'manual_csv_path': PosixPath('/Users/dorianbesson/EDOC/IA/iapr2026do/project/training_data/object_labels/reference_cards/reference_manual.csv'),
 'components_csv_path': PosixPath('/Users/dorianbesson/EDOC/IA/iapr2026do/project/training_data/object_labels/reference_cards/reference_manual_components.csv')}

## Inspect reference_manual.csv

In [5]:
import pandas as pd

manual_csv = state["reference_manual_csv"]
df = pd.read_csv(manual_csv)
print(manual_csv)
df.head(20)

ModuleNotFoundError: No module named 'pandas'

## Quick Visual Check (Preview + Transparent PNG Sample)

In [ ]:
import matplotlib.pyplot as plt
import cv2

if not state["results"]:
    raise RuntimeError("No results available. Run the split cell first.")

first_name = next(iter(state["results"]))
res = state["results"][first_name]

if res.preview_path is not None and res.preview_path.is_file():
    preview_bgr = cv2.imread(str(res.preview_path), cv2.IMREAD_COLOR)
    preview_rgb = cv2.cvtColor(preview_bgr, cv2.COLOR_BGR2RGB)
else:
    preview_rgb = None

rgba_sample = None
if res.rgba_crop_paths:
    rgba_sample = cv2.imread(str(res.rgba_crop_paths[0]), cv2.IMREAD_UNCHANGED)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].set_title(f"Indexed Preview: {first_name}")
if preview_rgb is not None:
    axes[0].imshow(preview_rgb)
axes[0].axis("off")

axes[1].set_title("Transparent PNG Sample")
if rgba_sample is not None:
    rgba_sample = cv2.cvtColor(rgba_sample, cv2.COLOR_BGRA2RGBA)
    axes[1].imshow(rgba_sample)
axes[1].axis("off")

plt.tight_layout()